In [35]:
import torch

print(torch.__version__)

2.13.0+cpu


In [36]:
inputs = torch.tensor(
    [
        [0.43, 0.15, 0.89], # Your x1
        [0.55, 0.87, 0.66], # journey x2
        [0.57, 0.85, 0.64], # starts x3
        [0.22, 0.58, 0.33], # with x4
        [0.77, 0.25, 0.10], # one x5
        [0.05, 0.80, 0.55] # step x6
    ]
)

In [37]:
inputs[0]

tensor([0.4300, 0.1500, 0.8900])

In [38]:
inputs[1]

tensor([0.5500, 0.8700, 0.6600])

In [39]:
inputs.shape[1]

3

In [40]:
inputs.shape[0]

6

### Computing the attention weights of input[1] with respect to all the inputs

In [41]:
## Here we are calculating the attention score of the input[1] with respect to all the other inputs

query = inputs[1]

attention_score_2 = torch.empty(inputs.shape[0]) # attention_score_2 = [0,0,0,0,0,0]

for i, x_i in enumerate(inputs):
    attention_score_2[i] = torch.dot(x_i, query)

print(attention_score_2)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


In [42]:
print(attention_score_2.sum())

tensor(6.5617)


In [43]:
res = 0.

for idx, element in enumerate(inputs[0]):
    res += inputs[0][idx] * query[idx]

print(res)
print(torch.dot(inputs[0], query))

tensor(0.9544)
tensor(0.9544)


In [44]:
## Now we need to use softmax for normalizartion perspective

attention_weights_2 = torch.softmax(attention_score_2, dim = 0)

print(f"Attention Weight 2: {attention_weights_2}")
print(f"Sum: {attention_weights_2.sum()}")

Attention Weight 2: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: 1.0


In [45]:
## Now we calculate the context vector with respect to inputs[1] which is the query
context_vec_2 = torch.zeros(query.shape)

for i, x_i in enumerate(inputs):
    context_vec_2 += attention_weights_2[i] * x_i

print(context_vec_2)

tensor([0.4419, 0.6515, 0.5683])


### A simple self attention mecahnism without trainable weights

#### Computing attention weights for all input tokens

In [46]:
inputs = torch.tensor(
    [
        [0.43, 0.15, 0.89], # Your x1
        [0.55, 0.87, 0.66], # journey x2
        [0.57, 0.85, 0.64], # starts x3
        [0.22, 0.58, 0.33], # with x4
        [0.77, 0.25, 0.10], # one x5
        [0.05, 0.80, 0.55] # step x6
    ]
)

In [47]:
## calculating the attention scores manually without matrix multiplication in pytorch which is much more optimized

attention_scores = torch.empty(6,6)

for i, x_i in enumerate(inputs):
    for j, x_j in enumerate(inputs):
        attention_scores[i, j] = torch.dot(x_i, x_j)

print(attention_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [48]:
## We will calculate the attention_scores using matrix multiplication in pytorch

attention_scores = inputs @ inputs.T
print(attention_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [49]:
# now we get the attention_weights with the softmax value of the attention scores

attention_weights = torch.softmax(attention_scores, dim = 1)

attention_weights

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])

In [50]:
attention_weights.sum(dim = 1)

tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])

In [51]:
# Now we calculate the context vectors

context_vectors = attention_weights @ inputs

context_vectors

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])

###  Implementing self attention with Trainable Weights

#### Computing the attention weights step by step

The context vector size does not depend on the input size

The below code is for **2nd input**

In [52]:
x_2 = inputs[1] # second input element
d_in = inputs.shape[1] # the input embedding size, d = 3
d_out = 2

In [53]:
x_2

tensor([0.5500, 0.8700, 0.6600])

In [54]:
torch.manual_seed(123)

W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad = True)
W_key = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad = True)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad = True)

print(W_query)
print(W_key)
print(W_value)

Parameter containing:
tensor([[0.2961, 0.5166],
        [0.2517, 0.6886],
        [0.0740, 0.8665]], requires_grad=True)
Parameter containing:
tensor([[0.1366, 0.1025],
        [0.1841, 0.7264],
        [0.3153, 0.6871]], requires_grad=True)
Parameter containing:
tensor([[0.0756, 0.1966],
        [0.3164, 0.4017],
        [0.1186, 0.8274]], requires_grad=True)


In [55]:
query_2 = x_2 @ W_query

print(query_2)


tensor([0.4306, 1.4551], grad_fn=<SqueezeBackward4>)


In [56]:
keys = inputs @ W_key

print(keys)

tensor([[0.3669, 0.7646],
        [0.4433, 1.1419],
        [0.4361, 1.1156],
        [0.2408, 0.6706],
        [0.1827, 0.3292],
        [0.3275, 0.9642]], grad_fn=<MmBackward0>)


In [57]:
value = inputs @ W_value

print(value)

tensor([[0.1855, 0.8812],
        [0.3951, 1.0037],
        [0.3879, 0.9831],
        [0.2393, 0.5493],
        [0.1492, 0.3346],
        [0.3221, 0.7863]], grad_fn=<MmBackward0>)


In [58]:
keys_2 = keys[1]

attention_score_22 = torch.dot(query_2, keys_2)

attention_score_22

tensor(1.8524, grad_fn=<DotBackward0>)

In [59]:
# calculation the attention_scores with respect to the 2nd input

attention_scores_2 = query_2 @ keys.T

attention_scores_2

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440],
       grad_fn=<SqueezeBackward4>)

In [60]:
dim_keys = keys.shape[1]

attention_weights_2 = torch.softmax(attention_scores_2 / dim_keys ** 0.5, dim = -1)

attention_weights_2

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820],
       grad_fn=<SoftmaxBackward0>)

In [61]:
attention_weights_2.sum()

tensor(1., grad_fn=<SumBackward0>)

In [62]:
# Now we need the context vectors with respect to the 2nd input

context_vectors_2 = attention_weights_2 @ value

context_vectors_2

tensor([0.3061, 0.8210], grad_fn=<SqueezeBackward4>)

The below code for the **entire inputs**

### Implementing Compact Self Attention

In [68]:
inputs = torch.tensor(
    [
        [0.43, 0.15, 0.89], # Your x1
        [0.55, 0.87, 0.66], # journey x2
        [0.57, 0.85, 0.64], # starts x3
        [0.22, 0.58, 0.33], # with x4
        [0.77, 0.25, 0.10], # one x5
        [0.05, 0.80, 0.55] # step x6
    ]
)

In [69]:
input_dimension = inputs.shape[1] # the input embedding size, d = 3
output_dimension = 2

In [70]:
keys_dimension = keys.shape[1]

In [71]:
import torch.nn as nn

class SelfAttention_v1(nn.Module):

    def __init__(self, input_dimension, output_dimension):
        super().__init__()
        self.W_query = torch.nn.Parameter(torch.rand(input_dimension, output_dimension))
        self.W_key = torch.nn.Parameter(torch.rand(input_dimension, output_dimension))
        self.W_value = torch.nn.Parameter(torch.rand(input_dimension, output_dimension))

    def forward(self, x):
        queries = inputs @ W_query
        keys = inputs @ W_key
        values = inputs @ W_value

        attention_scores = queries @ keys.T
        attention_weights = torch.softmax(attention_scores / keys_dimension**0.5, dim = 1)

        context_vectors = attention_weights @ values

        return context_vectors

torch.manual_seed(123)
self_attention_v1 = SelfAttention_v1(d_in, d_out)

In [72]:
self_attention_v1(inputs)

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)

In Pytorch we have a better way to do this and it is to use Linear layers. This has weights and bias. In our case we need the weights.

In [73]:
import torch.nn as nn

class SelfAttention_v2(nn.Module):

    def __init__(self, input_dimension, output_dimension, qkv_bias = False):

        super().__init__()

        self.W_query = torch.nn.Linear(input_dimension, output_dimension, bias = qkv_bias)
        self.W_key = torch.nn.Linear(input_dimension, output_dimension, bias = qkv_bias)
        self.W_value = torch.nn.Linear(input_dimension, output_dimension, bias = qkv_bias)

    def forward(self, x):

        queries = self.W_query(inputs)
        keys = self.W_key(inputs)
        values = self.W_value(inputs)

        attention_scores = queries @ keys.T
        attention_weights = torch.softmax(attention_scores / keys_dimension**0.5, dim = 1)

        context_vectors = attention_weights @ values

        return context_vectors

torch.manual_seed(123)
self_attention_v2 = SelfAttention_v2(d_in, d_out)

In [74]:
self_attention_v2(inputs)

tensor([[-0.5337, -0.1051],
        [-0.5323, -0.1080],
        [-0.5323, -0.1079],
        [-0.5297, -0.1076],
        [-0.5311, -0.1066],
        [-0.5299, -0.1081]], grad_fn=<MmBackward0>)